# 02 — Temporal Dataset Construction and Splitting

This notebook composes the immutable modeling specifications, builds the canonical unsplit dataset from bronze storage, inspects the aligned runtime artifacts, and applies the fixed purged temporal splitter.

The dates and tickers are illustrative. Edit them to fit the history available in your local database while preserving non-overlapping train, validation, and test ranges.

In [ ]:
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"
TICKERS = ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
DATA_CUTOFF = date(2025, 12, 31)

TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2021, 12, 31)
VALIDATION_START = date(2022, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

## Define the unsplit dataset specification

`TemporalDatasetSpec` contains only choices that determine the unsplit data product. Split dates, model configuration, random seeds, and tracking are deliberately absent. This notebook creates a compact `FeatureSetSpec`, a resolved `UniverseSpec`, and an explicit `SupervisedTaskSpec`; production experiments would normally reuse a catalog task unchanged.

In [ ]:
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import (
    SupervisedTaskSpec,
    TemporalDatasetSpec,
    UniverseSpec,
    V2_PRIMARY_TASK,
    V2_TARGET_SET,
)

feature_set = DEFAULT_FEATURE_SET.select(
    "returns",
    "trend",
    "volatility",
    "volume",
    name="notebook_ohlcv_features",
    version="1",
)

universe = UniverseSpec(
    name="notebook_training_universe",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)

task_spec = SupervisedTaskSpec(
    name="notebook_tp_before_sl_5d_classification",
    target_set_name=V2_TARGET_SET.name,
    target_set_version=V2_TARGET_SET.version,
    target_column=V2_PRIMARY_TASK.target_column,
    task_type=V2_PRIMARY_TASK.task_type,
    horizon_sessions=V2_PRIMARY_TASK.horizon_sessions,
    target_end_date_column=V2_PRIMARY_TASK.target_end_date_column,
)

dataset_spec = TemporalDatasetSpec(
    feature_set=feature_set,
    target_set=V2_TARGET_SET,
    task=task_spec,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
)

print(dataset_spec.digest)
print(dataset_spec.to_json())

## Build the canonical unsplit bundle

`build_temporal_dataset()` loads the required bronze columns, evaluates training eligibility at the cutoff, and delegates the in-memory feature, target, alignment, and manifest work to `construct_temporal_dataset()`.

In [ ]:
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import build_temporal_dataset

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
engine = resolve_database_engine(database_url=database_url)
bundle = build_temporal_dataset(engine=engine, spec=dataset_spec)

bundle.manifest.to_manifest()

The three DataFrames use the same canonical sample index. The selected target is complete, while feature warm-up missing values remain visible.

In [ ]:
assert bundle.features.index.equals(bundle.targets.index)
assert bundle.features.index.equals(bundle.samples.index)
assert bundle.targets[task_spec.target_column].notna().all()

{
    "rows": len(bundle.features),
    "feature_columns": len(bundle.features.columns),
    "target_columns": len(bundle.targets.columns),
    "sample_columns": list(bundle.samples.columns),
}

In [ ]:
bundle.samples.head()

## Create the split specification

Every date range is inclusive and shared across tickers. Purging uses each row's actual `target_end_date`; an optional embargo removes additional observed signal dates from the end of train and validation after purging.

In [ ]:
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    ModelSpec,
    TemporalSplitSpec,
)

split_spec = TemporalSplitSpec(
    name="notebook_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
    embargo_sessions=0,
)

split_spec.to_manifest()

## Compose the complete experiment specification

`ModelSpec` records a concrete implementation path and JSON-compatible hyperparameters. `ExperimentSpec` binds the dataset choices, split policy, model configuration, and random seeds. Its `dataset_spec` property reconstructs the lower-level unsplit specification used above; it does not store a second independent copy.

In [ ]:
model_spec = ModelSpec(
    name="logistic_regression_baseline",
    version="1",
    model_type="sklearn.linear_model.LogisticRegression",
    hyperparameters={
        "max_iter": 1_000,
        "class_weight": "balanced",
        "random_state": 42,
    },
)

experiment_spec = ExperimentSpec(
    name="notebook_v2_baseline",
    version="1",
    feature_set=feature_set,
    target_set=V2_TARGET_SET,
    task=task_spec,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
    split=split_spec,
    model=model_spec,
    random_seeds={"model": 42},
)

assert experiment_spec.dataset_spec.digest == dataset_spec.digest
print(experiment_spec.digest)

## Assign train, validation, and locked-test positions

`TemporalSplitResult` retains an auditable annotation for every row in the unsplit bundle. Its positional indices refer back to the aligned bundle frames.

In [ ]:
from swingtrader.modeling.experiments import FixedTemporalSplitter

splitter = FixedTemporalSplitter(experiment_spec.split)
split_result = splitter.assign(bundle)

train_index = split_result.indices("train")
validation_index = split_result.indices("validation")
locked_test_index = split_result.indices("test")

{
    "train": split_result.summary("train").to_manifest(),
    "validation": split_result.summary("validation").to_manifest(),
}

In [ ]:
print(split_result.manifest.digest)

In [ ]:
split_result.samples[["split", "split_exclusion_reason"]].value_counts(dropna=False)

## Extract aligned model inputs

The tabular adapter performs no preprocessing. Imputation, feature selection, scaling, and model-specific conversion belong inside the future training workflow and must be fitted on training rows only.

In [ ]:
from swingtrader.modeling.datasets import to_tabular_dataset

tabular = to_tabular_dataset(bundle)

X_train = tabular.X.iloc[train_index]
y_train = tabular.y.iloc[train_index]
X_validation = tabular.X.iloc[validation_index]
y_validation = tabular.y.iloc[validation_index]

{
    "train": X_train.shape,
    "validation": X_validation.shape,
}

Do not materialize or inspect locked-test features or targets while selecting preprocessing, model hyperparameters, or decision thresholds. `locked_test_index` must only be consumed by the later final-evaluation workflow. A future baseline-model notebook should be added when the repository implements that split-aware training workflow.